In [ ]:
# SPDX-License-Identifier: MIT
# SPDX-FileCopyrightText: 2026 CEA and ANSES

# Processsing of *L. monocytogenes* and *V. parahaemolyticus* Raman spectra with Raman-DIP for their metabolic signature study 

In [ ]:
import pickle
from __future__ import annotations

# Preprocessed spectra import 

In [ ]:
with open("preprocess_saves/2025-12-18_12strains_als-lam1e5-p0.01-niter10-zeroClip_cosmic-multispectra-zthresh5.0_norm-l2.pkl", "rb") as f:
    preprocessed_spectrum = pickle.load(f)

# Selection of the processing windows

In [ ]:
from src.utils import select_windows
from src.utils import select_two_windows_dict
from src.utils import calculate_area_ratio_dicts
from src.utils import crop_spectra_dict
from src.utils import combiner_groupes

from src.models import pca_dict
from src.models import gridsearch_logreg_dict
from src.models import analyse_modele_dict_v3
from src.models import gridsearch_classif_dict_v2

from src.visualization import boxplot_ratio_and_median_spectra_v2
from src.visualization import boxplot_pca_dict_matplotlib_v2
from src.visualization import plot_group_median_dict
from src.visualization import pca_loadings_and_median_spectra

In [ ]:
reduced_dict_fp = {}

windows_to_select = [(600, 1730)]  

for key, df in preprocessed_spectrum.items():
    df_reduced = select_windows(df, windows_to_select)
    reduced_dict_fp[key] = df_reduced

In [ ]:
reduced_dict_cdr = {}

windows_to_select = [(1800, 3050)]  

for key, df in preprocessed_spectrum.items():
    df_reduced = select_windows(df, windows_to_select)
    reduced_dict_cdr[key] = df_reduced

In [ ]:
sampeled_dict = {
    key: (df.sample(n=35, random_state=42) if len(df) > 35 else df)
    for key, df in reduced_dict_fp.items()
}

sampeled_dict_CD = {
    key: (df.sample(n=35, random_state=42) if len(df) > 35 else df)
    for key, df in reduced_dict_cdr.items()
}

In [ ]:
plot_group_median_dict(sampeled_dict, title="Median spectra after preprocessing and sampling")   

In [ ]:
plot_group_median_dict(reduced_dict_cdr, title="Median spectra after preprocessing and sampling") 

## CD band analysis

In [ ]:
CD_selected, CH_selected = select_two_windows_dict(
    reduced_dict_cdr,
    window1=(2075, 2250),
    window2=(2800, 3000)
)

In [ ]:
ratio_dict = calculate_area_ratio_dicts(CD_selected, CH_selected)

In [ ]:
zoom_cd = crop_spectra_dict(CD_selected, 2030, 2300)

In [ ]:
rennomage = {
    "temoin": "Control",
    "0025topax": "0.025 % Topax",
    "05topactive": "0.5 % Topactive",
    "025topactive": "0.25 % Topactive",
}

In [ ]:
selection_tout = ["EGDe", "ScottA", "16Li100A", "L6202V1", "16Li73B", "MAN195TJ01",
                "ATCC17802", "18-1732", "18-1890", "19-0924", "20-2383", "21-2854"]
label_suppr = ["100chauffe"]

selection_deux_ref = ["EGDe", "ATCC17802"]

boxplot_ratio_and_median_spectra_v2(
    ratio_dict,
    spectra_median_dict=zoom_cd,
    title="",
    ylabel="$\\it{{Ratio}}$ CD/(CD+CH)",
    figsize=(10, 50),
    label_map=rennomage,
    exclude_labels=label_suppr,
    prefix_selection=selection_tout,
    control_labels=["control", "temoin"],
    stats_bracket_spacing=0.7,
    colors=["#000033", "#3366CC", "#99CCFF", "#CCCCFF"],
)

In [ ]:
pc_cd = pca_dict(CD_selected, pc_pair=[0, 1], n_components_total=10, meta_index=0, pattern_sep='_')

In [ ]:
pc_numbers = [2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2]

In [ ]:
label_suppr = ["100chauffe"]
ordre_prefix = ["EGDe", "ScottA", "16Li100A", "L6202V1", "16Li73B", "MAN195TJ01",
                "ATCC17802", "18-1732", "18-1890", "19-0924", "20-2383", "21-2854"]

colors = ["#000033", "#3366CC", "#99CCFF"]

boxplot_pca_dict_matplotlib_v2(
    pc_cd,
    pc_numbers=pc_numbers,
    title="",
    ylabel="Selected Prinbcipal Component values",
    figsize=(16, 20),
    label_map=rennomage,
    exclude_labels=label_suppr,
    label_order=list(rennomage.values()),
    group_order=ordre_prefix,
    colors=colors
)

In [ ]:
pca_loadings_and_median_spectra(
    pc_cd,
    spectra_median_dict=zoom_cd,
    pc_pair=[0],
    prefix_selection=["EGDe", "ScottA", "16Li100A", "L6202V1", "16Li73B", "MAN195TJ01",
                       "ATCC17802", "18-1732", "18-1890", "19-0924",
                       "20-2383", "21-2854"],
    label_map=rennomage,
    exclude_labels=label_suppr,
    figsize=(10, 40),
    tick_step=5,
    colors=["#000033", "#3366CC", "#99CCFF", "#CCCCFF"],
)

## Machine Learning benchmarks on fingerprint region

In [ ]:
best_results = gridsearch_classif_dict_v2(
    sampeled_dict,
    meta_index=0,
    pattern_sep='_',
    test_size=0.2,
    random_state=42,
    scoring="accuracy",
    cv=5,
    exclude_labels=label_suppr
    )

## Gridsearch of best parameters using Logistic Regression

In [ ]:
listeria = combiner_groupes(sampeled_dict, ['EGDe', 'ScottA', '16Li100A', 'L6202V1', '16Li73B', 'MAN195TJ01'], "Lm")
vibrio = combiner_groupes(sampeled_dict, ['ATCC17802', '18-1732', '18-1890', '19-0924', '20-2383', '21-2854'], "Vp")

strain_dict = {**listeria, **vibrio}

In [ ]:
best_model_logreg_lm_vp = gridsearch_logreg_dict(strain_dict,
    meta_index=0,
    pattern_sep='_',
    test_size=0.2,
    random_state=42,
    scoring="accuracy",
    exclude_labels=label_suppr,
    cv=10,
    smooth_window=11,
    smooth_poly=2,
    tick_spacing_x=10,
    tick_spacing_y=0.1
    )

## Best parameters of Logistic Regression train/test

In [ ]:
remapping_names = {
    "temoin": "Control",
    "0025topax": "0.025 % Topax",
    "05topactive": "0.5 % Topactive",
    "025topactive": "0.25 % Topactive",
}

rapport_logreg = analyse_modele_dict_v3(
    strain_dict,
    best_model_logreg_lm_vp,
    meta_index=0,
    pattern_sep='_',
    do_smooth=False,
    smooth_window=20,
    smooth_poly=5,
    tick_spacing_x=10,
    tick_spacing_y=0.1,
    exclude_labels=label_suppr,
    top_features=20,
    sensi_peakfinder=0.4,  # 0.4
    class_name_map=remapping_names
    )